In [1]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
seed_number = 42
import random

random.seed(seed_number)
np.random.seed(seed_number)

In [4]:
datasetName = 'Heloc'

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
import dice_ml
from dice_ml.utils import helpers

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

dataset = pd.read_csv(f"{ugce_dir}/data/heloc.csv")
dataset = dataset[(dataset.iloc[:, 1:] >= 0).all(axis=1)]
dataset = dataset.reset_index(drop=True)
first_column = dataset.pop(dataset.columns[0])
TARGET_COLUMN = "RiskPerformance"
dataset[TARGET_COLUMN] = first_column
dataset[TARGET_COLUMN] = LabelEncoder().fit_transform(dataset[TARGET_COLUMN])
target = dataset[TARGET_COLUMN]

datasetX = dataset.copy()
datasetX = datasetX.drop(columns=[TARGET_COLUMN])

x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = datasetX.columns.to_list()
categorical = x_train.columns.difference(numerical)

try:
    import joblib
    model = joblib.load(f"{ugce_dir}/results/models/{datasetName}_model.pkl")
except:
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    transformations = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical),
            ('cat', categorical_transformer, categorical)])

    model = RandomForestClassifier(random_state=42)

    model = Pipeline(steps=[('preprocessor', transformations),
                        ('classifier', model)])

    model.fit(x_train, y_train)

    import joblib
    os.makedirs(f"{ugce_dir}/results/models", exist_ok=True)
    joblib.dump(model, f"{ugce_dir}/results/models/{datasetName}_model.pkl")

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

negative_instances = x_test[model.predict(x_test) == 0]
instances_to_explain = negative_instances
print("Number of instances to explain: ", len(instances_to_explain))

Accuracy:  0.688622754491018
Number of instances to explain:  368


In [6]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

iea.dataset.describe()

,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,MaxDelq2PublicRecLast12M,...,PercentInstallTrades,MSinceMostRecentInqexcl7days,NumInqLast6M,NumInqLast6Mexcl7days,NetFractionRevolvingBurden,NetFractionInstallBurden,NumRevolvingTradesWBalance,NumInstallTradesWBalance,NumBank2NatlTradesWHighUtilization,PercentTradesWBalance
count,2502.00000,2502.000000,2502.000000,2502.000000,2502.000000,2502.000000,2502.000000,2502.000000,2502.000000,2502.000000,...,2502.000000,2502.000000,2502.000000,2502.000000,2502.000000,2502.000000,2502.000000,2502.000000,2502.000000,2502.000000
mean,66.29976,204.900480,7.059552,76.457234,24.232614,0.958833,0.578737,87.091926,21.116307,4.720224,...,38.375300,2.118305,1.673062,1.611111,41.115108,68.542366,4.761391,3.120304,1.277378,70.717426
std,7.83142,91.778897,6.514160,26.844562,11.208800,1.483046,1.168011,10.837221,20.709788,1.598199,...,15.286241,4.255203,2.142893,2.090573,27.639688,23.567567,3.131948,1.735486,1.498700,18.773820
min,36.00000,21.000000,0.000000,13.000000,1.000000,0.000000,0.000000,20.000000,0.000000,0.000000,...,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,10.000000
25%,61.00000,140.000000,3.000000,59.000000,16.000000,0.000000,0.000000,82.000000,4.000000,4.000000,...,27.000000,0.000000,0.000000,0.000000,17.000000,54.000000,3.000000,2.000000,0.000000,57.000000
50%,66.00000,189.000000,5.000000,74.000000,23.000000,1.000000,0.000000,90.000000,13.000000,5.000000,...,38.000000,0.000000,1.000000,1.000000,39.000000,73.000000,4.000000,3.000000,1.000000,71.000000
75%,72.00000,261.000000,9.000000,92.000000,30.000000,1.000000,1.000000,95.000000,33.000000,6.000000,...,49.000000,2.000000,2.000000,2.000000,61.750000,87.000000,6.000000,4.000000,2.000000,85.000000
max,89.00000,604.000000,65.000000,224.000000,78.000000,16.000000,16.000000,99.000000,83.000000,7.000000,...,92.000000,24.000000,24.000000,24.000000,154.000000,153.000000,32.000000,23.000000,12.000000,100.000000


In [7]:
pd.set_option('display.max_columns', None)
numerical_columns = iea.dataset.select_dtypes(include=['int64', 'float64']).columns

non_zero_descriptions = {}

for col in numerical_columns:
    non_zero_values = iea.dataset[iea.dataset[col] != 0][col]
    if not non_zero_values.empty:
        non_zero_descriptions[col] = non_zero_values.describe()

# Display results
for feature, stats in non_zero_descriptions.items():
    print(f"\n Feature: {feature}")
    print(stats)


 Feature: ExternalRiskEstimate
count    2502.00000
mean       66.29976
std         7.83142
min        36.00000
25%        61.00000
50%        66.00000
75%        72.00000
max        89.00000
Name: ExternalRiskEstimate, dtype: float64

 Feature: MSinceOldestTradeOpen
count    2502.000000
mean      204.900480
std        91.778897
min        21.000000
25%       140.000000
50%       189.000000
75%       261.000000
max       604.000000
Name: MSinceOldestTradeOpen, dtype: float64

 Feature: MSinceMostRecentTradeOpen
count    2474.000000
mean        7.139450
std         6.507232
min         1.000000
25%         3.000000
50%         5.000000
75%         9.000000
max        65.000000
Name: MSinceMostRecentTradeOpen, dtype: float64

 Feature: AverageMInFile
count    2502.000000
mean       76.457234
std        26.844562
min        13.000000
25%        59.000000
50%        74.000000
75%        92.000000
max       224.000000
Name: AverageMInFile, dtype: float64

 Feature: NumSatisfactoryTrades
cou

In [ ]:
feat_unique = {}
for col in datasetX.columns:
    feat_unique[col] = len(datasetX[col].unique())
feat_unique
## sort the features by the number of unique values
top_3_difficult_to_change_cols = sorted(feat_unique.items(), key=lambda x: x[1])
top_3_difficult_to_change_cols

[('MaxDelqEver', 5),
 ('MaxDelq2PublicRecLast12M', 8),
 ('NumBank2NatlTradesWHighUtilization', 12),
 ('NumTrades90Ever2DerogPubRec', 14),
 ('NumTrades60Ever2DerogPubRec', 15),
 ('NumTradesOpeninLast12M', 15),
 ('NumInstallTradesWBalance', 16),
 ('NumInqLast6Mexcl7days', 18),
 ('NumInqLast6M', 20),
 ('NumRevolvingTradesWBalance', 24),
 ('MSinceMostRecentInqexcl7days', 25),
 ('MSinceMostRecentTradeOpen', 46),
 ('ExternalRiskEstimate', 49),
 ('PercentTradesNeverDelq', 62),
 ('NumSatisfactoryTrades', 69),
 ('NumTotalTrades', 77),
 ('PercentTradesWBalance', 79),
 ('MSinceMostRecentDelq', 84),
 ('PercentInstallTrades', 85),
 ('NetFractionRevolvingBurden', 114),
 ('NetFractionInstallBurden', 118),
 ('AverageMInFile', 152),
 ('MSinceOldestTradeOpen', 419)]

# Constraints Type Series:
1. Immutability
2. Ranges
3. Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        'MaxDelqEver': 'i',
        'MaxDelq2PublicRecLast12M': 'i'
    },
    2:{
        'MaxDelqEver': 'i',
        'MaxDelq2PublicRecLast12M': 'i',
        'PercentTradesNeverDelq': (20, 60),
        'NumSatisfactoryTrades': (1, 20),
        
    },
    3: {
        'MaxDelqEver': 'i',
        'MaxDelq2PublicRecLast12M': 'i',
        'PercentTradesNeverDelq': (20, 60),
        'NumSatisfactoryTrades': (1, 20),
        "MSinceOldestTradeOpen": 'decr',
        "NumSatisfactoryTrades": 'incr'
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

results_incremental_imm_ranges_direct_arr = []
import time
strategy = "fix_population_update_fitness"
for i in range(5):
    results_incremental_imm_ranges_direct = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_imm_ranges_direct_arr.append(results_incremental_imm_ranges_direct)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_imm_ranges_direct_arr, open(f"{results_dir}/results_incremental_imm_ranges_direct_arr.pkl", "wb"))

100%|██████████| 368/368 [02:28<00:00,  2.49it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [02:30<00:00,  2.45it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [02:23<00:00,  2.57it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [02:27<00:00,  2.49it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [02:28<00:00,  2.48it/s]

Empty intermediate counter: 0


In [9]:
strategy = "fix_population_update_fitness"

In [ ]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_imm_ranges_direct = pickle.load(open(f'{results_dir}/results_incremental_imm_ranges_direct_arr.pkl', 'rb'))

# Constraints Type Series:
1. Ranges
2. Immutability
3. Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        'PercentTradesNeverDelq': (20, 60),
        'NumSatisfactoryTrades': (1, 20)
    },
    2:{
        'PercentTradesNeverDelq': (20, 60),
        'NumSatisfactoryTrades': (1, 20),
        'MaxDelqEver': 'i',
        'MaxDelq2PublicRecLast12M': 'i'
        
        
    },
    3: {
        'PercentTradesNeverDelq': (20, 60),
        'NumSatisfactoryTrades': (1, 20),
        'MaxDelqEver': 'i',
        'MaxDelq2PublicRecLast12M': 'i',
        "MSinceOldestTradeOpen": 'decr',
        "NumSatisfactoryTrades": 'incr'
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

results_incremental_ranges_imm_incr_arr = []
import time
strategy = "fix_population_update_fitness"
for i in range(5):
    results_incremental_ranges_imm_incr = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_ranges_imm_incr_arr.append(results_incremental_ranges_imm_incr)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_ranges_imm_incr_arr, open(f"{results_dir}/results_incremental_ranges_imm_incr_arr.pkl", "wb"))

100%|██████████| 368/368 [02:22<00:00,  2.59it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [02:21<00:00,  2.59it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [02:21<00:00,  2.60it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [02:20<00:00,  2.61it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [02:24<00:00,  2.55it/s]

Empty intermediate counter: 0


In [ ]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_ranges_imm_direct = pickle.load(open(f'{results_dir}/results_incremental_ranges_imm_incr_arr.pkl', 'rb'))

# Constraints Type Series:
1. Directionality
2. Immutability
3. Ranges

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        "MSinceOldestTradeOpen": 'decr',
        "NumSatisfactoryTrades": 'incr'
    },
    2:{
        'PercentTradesNeverDelq': (20, 60),
        'NumSatisfactoryTrades': (1, 20),
        'MaxDelqEver': 'i',
        'MaxDelq2PublicRecLast12M': 'i'
    },
    3: {
        "MSinceOldestTradeOpen": 'decr',
        "NumSatisfactoryTrades": 'incr',
        'MaxDelqEver': 'i',
        'MaxDelq2PublicRecLast12M': 'i',
        'PercentTradesNeverDelq': (20, 60),
        'NumSatisfactoryTrades': (1, 20)
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

import time
strategy = "fix_population_update_fitness"
results_incremental_dir_im_range_arr = []
for i in range(5):
    results_incremental_dir_im_range = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_dir_im_range_arr.append(results_incremental_dir_im_range)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_dir_im_range_arr, open(f"{results_dir}/results_incremental_dir_im_range_arr.pkl", "wb"))

100%|██████████| 368/368 [02:29<00:00,  2.46it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [02:23<00:00,  2.56it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [02:28<00:00,  2.48it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [02:27<00:00,  2.49it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [02:31<00:00,  2.43it/s]

Empty intermediate counter: 0


In [ ]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_dir_im_range = pickle.load(open(f'{results_dir}/results_incremental_dir_im_range_arr.pkl', 'rb'))

# Make Plots

In [ ]:
from test_utils import gather_results_sequence_of_type_constraints
import matplotlib.pyplot as plt

constraint_orders = ["I→R→D", "R→I→D", "D→I→R"]

time_dynamic_imm_ranges_direct, avg_generations_imm_ranges_direct, avg_cfes_found_imm_ranges_direct, avg_proximity_loss_imm_ranges_direct, avg_sparsity_imm_ranges_direct, avg_intermediate_imm_ranges_incr, \
time_dynamic_ranges_imm_incr, avg_generations_ranges_imm_incr, avg_cfes_found_ranges_imm_incr, avg_proximity_loss_ranges_imm_incr, avg_sparsity_ranges_imm_incr, avg_intermediate_ranges_imm_incr, \
time_dynamic_dir_im_range, avg_generations_dir_im_range, avg_cfes_found_dir_im_range, avg_proximity_loss_dir_im_range, avg_sparsity_dir_im_range, avg_intermediate_dir_im_range =\
    gather_results_sequence_of_type_constraints(iea, results_incremental_imm_ranges_direct_arr, results_incremental_ranges_imm_incr_arr, results_incremental_dir_im_range_arr, verbose=True) 

cfe_found = [
        avg_cfes_found_imm_ranges_direct,
        avg_cfes_found_ranges_imm_incr,
        avg_cfes_found_dir_im_range
]
avg_time = [
    time_dynamic_imm_ranges_direct,
    time_dynamic_ranges_imm_incr,
    time_dynamic_dir_im_range
]
avg_weighted_l1 = [
    avg_proximity_loss_imm_ranges_direct,
    avg_proximity_loss_ranges_imm_incr,
    avg_proximity_loss_dir_im_range
]

avg_sparsity = [
    avg_sparsity_imm_ranges_direct,
    avg_sparsity_ranges_imm_incr,
    avg_sparsity_dir_im_range
]
results = {
    "cfe_found": cfe_found,
    "avg_time": avg_time,
    "avg_weighted_l1": avg_weighted_l1,
    "avg_sparsity": avg_sparsity
}

In [ ]:
table_data = pd.DataFrame(results, index=constraint_orders)
print(table_data.to_latex(float_format="%.4f"))

\begin{tabular}{lrrrr}
\toprule
 & cfe_found & avg_time & avg_weighted_l1 & avg_sparsity \\
\midrule
I→R→D & 99.4595 & 1.7911 & 0.0515 & 0.0305 \\
R→I→D & 99.4595 & 1.7242 & 0.0514 & 0.0305 \\
D→I→R & 99.4595 & 1.7974 & 0.0535 & 0.0309 \\
\bottomrule
\end{tabular}

